# 01. Island sizes

**Tier:** Deep. **Why you'd use this:** A grid marks land with `"1"` and water with `"0"`. Connected land (up, down, left, right) forms an island. You want every island's area, sorted.

**After this lesson you can:** flood-fill from each unvisited land cell; count cells while sinking an island; collect sizes.
About 25 minutes. **Uses:** [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Recursion](../../course/09-recursion/01-base-case-lesson.ipynb)


## Quick review

1. How do you index a cell in a list of rows?
2. What stops a recursive flood into water or off the grid?

<details><summary>Answers</summary>

`grid[row][col]`. A bounds check or a water / already-seen cell returns without recursing. Skim [Recursion](../../course/09-recursion/01-base-case-lesson.ipynb) if deep calls feel unclear.
</details>


## Recognize it

A 2-D grid of land and water, and a question about connected components of land. The prompt says "number of islands", "island perimeter", or "sizes of islands".

Signals: 4-direction adjacency; mark visited (or sink) so you do not recount. A look-alike that is not this pattern: shortest path in a grid with costs — that needs a queue and distances, not only a flood count.


## Why it works this way

Each land cell belongs to exactly one island. Starting a flood from an unseen land cell visits every cell in that component by walking to orthogonal neighbors. Marking a cell as water (or seen) as you enter it prevents revisiting. The count of visits in one flood is the island's size.

A nested scan that, for every land cell, explores the whole island without marking would re-explore the same cells from every starting point inside the island — work proportional to size squared per island. Marking makes each cell enter a flood at most once across the whole grid, so total time is proportional to rows times columns.

Recursion uses the call stack as the frontier. An explicit queue does the same walk iteratively and avoids depth limits on huge islands. Diagonal neighbors are a different problem statement; this sibling uses only up, down, left, right. Copying the grid before mutating keeps the caller's input intact when tests reuse the same literal.
Number-of-islands is the same flood with a counter instead of a size list. Perimeter problems add edge counting while you walk. Rotting oranges run a multi-source BFS from every rotten cell at once so minutes equal distance. Word search is a flood on a character grid with backtracking that unmarks on the way out.

Grid graphs are implicit: you never store an adjacency list of every cell; neighbors are computed from coordinates. That keeps memory proportional to the frontier and the seen set. On huge maps, iterative BFS avoids recursion limits when a single island covers the whole board.


## When to use it

Use flood fill when you need component counts or sizes on a grid. Prefer BFS with distances when the question is shortest path. Prefer Union-Find when you merge cells online as edges arrive. Prefer a plain cell count when connectivity does not matter.
 Hex grids change the neighbor list; the flood idea is unchanged once you enumerate the correct offsets.


## Brute force, and why it's slow

From every land cell, flood the whole island with a fresh `seen` set. Each island of size `s` is fully re-walked `s` times.


In [1]:
def explore(grid, row, col, rows, cols, seen):
    if (
        not (0 <= row < rows and 0 <= col < cols)
        or grid[row][col] != "1"
        or (row, col) in seen
    ):
        return 0
    seen.add((row, col))
    return (
        1
        + explore(grid, row + 1, col, rows, cols, seen)
        + explore(grid, row - 1, col, rows, cols, seen)
        + explore(grid, row, col + 1, rows, cols, seen)
        + explore(grid, row, col - 1, rows, cols, seen)
    )


def island_touch_count(grid):
    if not grid or not grid[0]:
        return 0
    rows, cols = len(grid), len(grid[0])
    touches = 0
    for row in range(rows):
        for col in range(cols):
            if grid[row][col] == "1":
                touches += explore(grid, row, col, rows, cols, set())
    return touches


sample = [
    ["1", "1", "0"],
    ["1", "1", "0"],
    ["0", "0", "1"],
]
print(island_touch_count(sample))


17


## Key insight

Sink (or mark) each land cell when you first visit it. One flood per island visits each of its cells once; the visit count is the size.


## Worked example

Return the sizes of all islands, sorted ascending. Land is `"1"`, water is `"0"`. Connections are orthogonal only.


In [2]:
def island_sizes(grid: list[list[str]]) -> list[int]:
    if not grid or not grid[0]:
        return []
    rows, cols = len(grid), len(grid[0])
    # 1. Copy so sinking does not mutate the caller's grid.
    grid = [row[:] for row in grid]

    def measure(row: int, col: int) -> int:
        # 2. Water, out of bounds, or already sunk → zero.
        if not (0 <= row < rows and 0 <= col < cols) or grid[row][col] != "1":
            return 0
        grid[row][col] = "0"
        return (
            1
            + measure(row + 1, col)
            + measure(row - 1, col)
            + measure(row, col + 1)
            + measure(row, col - 1)
        )

    sizes = []
    # 3. Every remaining "1" starts a new island.
    for row in range(rows):
        for col in range(cols):
            if grid[row][col] == "1":
                sizes.append(measure(row, col))
    return sorted(sizes)


sample = [
    ["1", "1", "0", "0", "0"],
    ["1", "1", "0", "0", "0"],
    ["0", "0", "1", "0", "0"],
    ["0", "0", "0", "1", "1"],
]
print(island_sizes([row[:] for row in sample]))
print(island_sizes([]))
print(island_sizes([["0"]]))


[1, 2, 4]
[]
[]


## Trace

Sample grid islands: a 2×2 block (size 4), a single cell (size 1), a two-cell bar (size 2).

| start (row, col) | measure returns | sizes so far |
|---|---|---|
| (0, 0) | 4 | `[4]` |
| (2, 2) | 1 | `[4, 1]` |
| (3, 3) | 2 | `[4, 1, 2]` |

Sorted → **`[1, 2, 4]`**.


## Cost

Each cell is written at most once: time grows with rows × columns. Extra memory is the recursion depth (island size) plus the copy of the grid. Unmarked re-exploration blows up with island size.

Sorting the size list at the end costs `k log k` for `k` islands, which is dominated by the grid scan when islands are few. Returning sizes unsorted is enough when the caller only needs a multiset; this sibling sorts to make tests stable.


## Variants

Not solved here: [Number of Islands](../../challenges/problems/12-number-of-islands.ipynb), [Rotting Oranges](../../challenges/problems/32-rotting-oranges.ipynb), [Word Search](../../challenges/problems/36-word-search.ipynb).


## Common mistake

This bug is silent. The flood walks diagonally as well as orthogonally, so cells that only touch at a corner merge into one island. There is no traceback. The print is `[5]`, and the answer is `[1, 1, 1, 1, 1]`.


In [3]:
def island_sizes_wrong(grid):
    if not grid or not grid[0]:
        return []
    rows, cols = len(grid), len(grid[0])
    grid = [row[:] for row in grid]

    def measure(row, col):
        if not (0 <= row < rows and 0 <= col < cols) or grid[row][col] != "1":
            return 0
        grid[row][col] = "0"
        total = 1
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr == 0 and dc == 0:
                    continue
                total += measure(row + dr, col + dc)  # includes diagonals
        return total

    sizes = []
    for row in range(rows):
        for col in range(cols):
            if grid[row][col] == "1":
                sizes.append(measure(row, col))
    return sorted(sizes)


corner = [
    ["1", "0", "1"],
    ["0", "1", "0"],
    ["1", "0", "1"],
]
print(island_sizes_wrong(corner))


[5]


The fix only recurses to `(row±1, col)` and `(row, col±1)`. Orthogonal connectivity keeps diagonal-only touches as separate islands. Run `island_sizes` on that corner grid and you get `[1, 1, 1, 1, 1]`.


In [4]:
print(island_sizes([
    ["1", "0", "1"],
    ["0", "1", "0"],
    ["1", "0", "1"],
]))


[1, 1, 1, 1, 1]


## Python idioms

Copy rows with `row[:]` before sinking. An iterative version uses `collections.deque` as the frontier and a `seen` set when you must not mutate the grid.


In [5]:
from collections import deque


def largest_island(grid: list[list[str]]) -> int:
    if not grid or not grid[0]:
        return 0
    rows, cols = len(grid), len(grid[0])
    seen: set[tuple[int, int]] = set()
    best = 0
    for row in range(rows):
        for col in range(cols):
            if grid[row][col] != "1" or (row, col) in seen:
                continue
            queue = deque([(row, col)])
            seen.add((row, col))
            size = 0
            while queue:
                r, c = queue.popleft()
                size += 1
                for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                    in_bounds = 0 <= nr < rows and 0 <= nc < cols
                    if in_bounds and grid[nr][nc] == "1" and (nr, nc) not in seen:
                        seen.add((nr, nc))
                        queue.append((nr, nc))
            best = max(best, size)
    return best


print(largest_island([
    ["1", "1", "0"],
    ["1", "0", "0"],
    ["0", "0", "1"],
]))


3


## Recap

- Each unseen land cell starts one flood.
- Sink or mark on entry so cells are counted once.
- Orthogonal neighbors only, unless the prompt says otherwise.

Next: [practice](01-island-sizes-practice.ipynb).
